✔ I/O 작업
→ ThreadPoolExecutor  
  
✔ CPU 작업  
→ multiprocessing  
  
✔ 고성능 비동기 I/O  
→ asyncio  

# 1. 멀티 쓰레드
- threading모듈을 사용
- CPython에는 GIL(Global Interpreter Lcok)이 있다.

In [2]:
import threading
import time

def worker():
    for i in range(3):
        print(f"Working... {i}")
        time.sleep(1)

t = threading.Thread(target=worker)  # 쓰레드 생성. 별도의 쓰레드로 실행할 함수를 지정
t.start()

print("Main thread continues")
t.join()  # 쓰레드 종료 대기
print("Done")

Working... 0Main thread continues

Working... 1
Working... 2
Done


In [3]:
# 여러 쓰레드를 실행
import threading
import time

def worker(name):
    for i in range(3):
        print(f"{name}: {i}")
        time.sleep(1)

threads = []

for i in range(3):
    t = threading.Thread(target=worker, args=(f"Thread-{i}",))
    t.start()
    threads.append(t)

for t in threads:
    t.join()

print("All threads finished")


Thread-0: 0Thread-1: 0
Thread-2: 0

Thread-1: 1Thread-2: 1
Thread-0: 1

Thread-2: 2Thread-1: 2
Thread-0: 2

All threads finished


### 공유 자원 문제 (Race Condition)  
멀티스레드의 가장 큰 문제는 공유 데이터 충돌입니다.

In [5]:
import threading

counter = 0

def increment():
    global counter
    for _ in range(100000):
        counter += 1  # 실제론 LOAD, ADD, STORE 3단계 연산

threads = [threading.Thread(target=increment) for _ in range(4)]

for t in threads:
    t.start()

for t in threads:
    t.join()

print(counter)  # 예상: 400000 ? 아닐 수도 있음. LOCK이 필요

400000


### Lock 사용 (동기화)
- 여러 명령을 하나로 묶어서 한번에 하나의 쓰레드만 수행하도록 Lock을 거는 것

In [7]:
import threading

counter = 0
lock = threading.Lock()

def increment():
    global counter
    for _ in range(100000):
        with lock:           # lock으로 잠금
            counter += 1

threads = [threading.Thread(target=increment) for _ in range(4)]

for t in threads:
    t.start()

for t in threads:
    t.join()

print(counter)  # 400000 보장

400000


| 도구        | 설명            |
| --------- | ------------- |
| Lock      | 기본 상호배제       |
| RLock     | 재진입 가능        |
| Semaphore | 동시 접근 개수 제한   |
| Event     | 신호 전달         |
| Condition | 상태 기반 대기      |
| Queue     | thread-safe 큐 |


### ThreadPoolExecutor
- 쓰레드 풀을 이용한 쓰레드 실행기
- 쓰레드 재사용
- 관리 자동화
- 예외 처리 쉬움
- Future 객체 제공

In [8]:
from concurrent.futures import ThreadPoolExecutor
import time

def task(n):
    time.sleep(1)
    return n * n

with ThreadPoolExecutor(max_workers=4) as executor:
    results = executor.map(task, range(5))

print(list(results))

[0, 1, 4, 9, 16]


# 2. 비동기 프로그래밍
- Python에서는 주로 asyncio 모듈과 async / await 문법을 사용

## 2.1 비동기란?
- 비동기 프로그래밍은 작업(특히 I/O) 이 완료되는 동안 기다리지 않고 다른 작업을 먼저 처리하는 방식이에요. 대기 시간이 긴 작업(예: 네트워크 요청, 파일 입출력)에서 특히 효율적이에요.

## 2.2 비동기 함수


| 개념         | 역할                    | 
| ---------- | ----------------------- |
| `async`    | 비동기 함수(코루틴) 정의          |         
| `await`    | 다른 비동기 작업 완료를 기다림       |      
| 이벤트 루프     | 비동기 작업 스케줄링 및 실행 담당     |
| asyncio 모듈 | Python 표준 비동기 I/O 프레임워크 |


In [9]:
# 비동기 함수 정의
async def say_hello():
    print("Hello")


In [ ]:
import asyncio

async def task(name, delay):
    print(f"{name} 시작")
    await asyncio.sleep(delay)  # 비동기 sleep
    print(f"{name} 완료")

async def main():
    await task("작업1", 2)
    await task("작업2", 1)

await main()

작업1 시작
작업1 완료
작업2 시작
작업2 완료


In [13]:
import asyncio

async def task(name, delay):
    print(f"{name} 시작")
    await asyncio.sleep(delay)
    print(f"{name} 완료")

async def main():
    await asyncio.gather(
        task("A", 2),
        task("B", 1),
        task("C", 3),
    )

await main()

A 시작
B 시작
C 시작
B 완료
A 완료
C 완료
